# CSIRO Biomass - Inference for T4×2 (Kaggle)

**最適化**: T4×2を活用した並列推論

## 戦略
1. GPU0: Fold 0, 2, 4のモデル
2. GPU1: Fold 1, 3のモデル
3. 交互に推論して最後にアンサンブル

In [ ]:
import os
import gc
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# T4×2の確認
n_gpus = torch.cuda.device_count()
print(f"Available GPUs: {n_gpus}")
for i in range(n_gpus):
    gpu = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i}: {gpu} ({vram:.1f} GB)")

# デバイス設定
device0 = torch.device("cuda:0" if n_gpus > 0 else "cpu")
device1 = torch.device("cuda:1" if n_gpus > 1 else "cuda:0")

In [ ]:
class CFG:
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    DATA_DIR = Path("/kaggle/input/csiro-biomass")
    MODEL_DIR = Path("/kaggle/input/csiro-dinov3-swa-v1")
    
    # T4用設定
    IMG_SIZE = 448  # T4メモリに合わせて調整
    N_FOLDS = 5
    BACKBONE = "vit_huge_plus_patch16_dinov3.lvd1689m"
    BATCH_SIZE = 1
    NUM_WORKERS = 2  # 並列処理用
    
    # GPU割り当て
    GPU0_FOLDS = [0, 2, 4]  # GPU0で処理
    GPU1_FOLDS = [1, 3]     # GPU1で処理

In [ ]:
# Load test data
test_df = pd.read_csv(CFG.DATA_DIR / "test.csv")
print(f"Test samples: {len(test_df)}")
test_wide = test_df[["image_path"]].drop_duplicates().reset_index(drop=True)
print(f"Unique test images: {len(test_wide)}")

In [ ]:
# Model definition
class LocalMambaBlock(nn.Module):
    def __init__(self, dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size//2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        x = x * torch.sigmoid(self.gate(x))
        x = self.dwconv(x.transpose(1, 2)).transpose(1, 2)
        x = self.proj(x)
        return shortcut + self.drop(x)


class BiomassModel(nn.Module):
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="")
        nf = self.backbone.num_features
        
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1),
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.head_green = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_dead = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_clover = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )

    def forward(self, x):
        left, right = x
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x = self.fusion(torch.cat([x_l, x_r], dim=1))
        x = self.pool(x.transpose(1, 2)).flatten(1)
        green = self.head_green(x)
        dead = self.head_dead(x)
        clover = self.head_clover(x)
        gdm = green + clover
        total = green + clover + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)

In [ ]:
# Dataset
class TestDataset(Dataset):
    def __init__(self, df, data_dir, transform):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row["image_path"]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        left = img.crop((0, 0, w // 2, h))
        right = img.crop((w // 2, 0, w, h))
        left = self.transform(left)
        right = self.transform(right)
        return left, right, row["image_path"]

def collate_fn(batch):
    lefts = torch.stack([b[0] for b in batch])
    rights = torch.stack([b[1] for b in batch])
    paths = [b[2] for b in batch]
    return lefts, rights, paths

test_tfms = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# 推論関数
def inference_fold(fold, device, test_loader):
    """単一Foldの推論"""
    model_path = CFG.MODEL_DIR / f"best_ema_fold{fold}.pth"
    if not model_path.exists():
        print(f"Fold {fold}: model not found")
        return None
    
    print(f"Fold {fold}: loading on {device}...")
    
    # メモリクリア
    if device.type == 'cuda':
        with torch.cuda.device(device):
            torch.cuda.empty_cache()
    
    model = BiomassModel(CFG.BACKBONE, pretrained=False)
    state_dict = torch.load(model_path, map_location="cpu")
    
    if list(state_dict.keys())[0].startswith("module."):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    preds = []
    with torch.no_grad():
        for left, right, _ in tqdm(test_loader, desc=f"Fold {fold} on {device}"):
            left = left.to(device)
            right = right.to(device)
            
            with torch.cuda.amp.autocast():
                out = model((left, right))
            
            preds.append(out.cpu().numpy())
    
    preds = np.vstack(preds)
    
    # モデル削除
    del model, state_dict
    if device.type == 'cuda':
        with torch.cuda.device(device):
            torch.cuda.empty_cache()
    gc.collect()
    
    return preds

In [ ]:
# T4×2並列推論
test_dataset = TestDataset(test_wide, CFG.DATA_DIR, test_tfms)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

FOLD_WEIGHTS = [1.0, 0.7, 0.9, 1.2, 0.9]
all_preds = []

# GPU0で処理
print(f"\n{'='*50}")
print(f"GPU 0: Processing folds {CFG.GPU0_FOLDS}")
print(f"{'='*50}")

for fold in CFG.GPU0_FOLDS:
    preds = inference_fold(fold, device0, test_loader)
    if preds is not None:
        all_preds.append((fold, preds * FOLD_WEIGHTS[fold]))
        print(f"Fold {fold}: shape={preds.shape}")

# GPU1で処理（2つ目のGPUがある場合）
if n_gpus > 1:
    print(f"\n{'='*50}")
    print(f"GPU 1: Processing folds {CFG.GPU1_FOLDS}")
    print(f"{'='*50}")
    
    for fold in CFG.GPU1_FOLDS:
        preds = inference_fold(fold, device1, test_loader)
        if preds is not None:
            all_preds.append((fold, preds * FOLD_WEIGHTS[fold]))
            print(f"Fold {fold}: shape={preds.shape}")
else:
    # 1GPUしかない場合は順次処理
    print(f"\n⚠️ Single GPU detected, processing remaining folds sequentially")
    for fold in CFG.GPU1_FOLDS:
        preds = inference_fold(fold, device0, test_loader)
        if preds is not None:
            all_preds.append((fold, preds * FOLD_WEIGHTS[fold]))

# ソートして順番を保証
all_preds.sort(key=lambda x: x[0])
preds_list = [p[1] for p in all_preds]
used_folds = [p[0] for p in all_preds]

print(f"\n{'='*50}")
print(f"Used folds: {used_folds}")
print(f"Total predictions: {len(preds_list)}")

In [ ]:
# アンサンブル
total_weight = sum([FOLD_WEIGHTS[f] for f in used_folds])
ensemble = np.sum(preds_list, axis=0) / total_weight
print(f"Ensemble: {ensemble.shape}")

# パスを取得
paths = []
for _, _, p in test_loader:
    paths.extend(p)

# Create predictions DataFrame
preds_wide = pd.DataFrame(ensemble, columns=CFG.TARGETS)
preds_wide.insert(0, 'image_path', paths)

# Convert to long format
preds_long = preds_wide.melt(
    id_vars=['image_path'],
    value_vars=CFG.TARGETS,
    var_name='target_name',
    value_name='target'
)

# Merge with test_df
submission = pd.merge(
    test_df[['sample_id', 'image_path', 'target_name']],
    preds_long,
    on=['image_path', 'target_name'],
    how='left'
)

submission = submission[['sample_id', 'target']]
submission['target'] = submission['target'].fillna(0.0).clip(lower=0)
submission = submission.sort_values('sample_id').reset_index(drop=True)

# Save
submission.to_csv("submission.csv", index=False)

print(f"\n✅ Saved: submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))
print(f"\nStats:")
print(submission['target'].describe())